# Price and basket, more than foot traffic

**Question.** At the Vasant Kunj market in New Delhi (12 April 2026), did the revenue ranking of the 27 brands come from more customers, more units per customer, or a higher price per unit?

**Data.** The organiser's leaderboard for each market: unique customers, units sold and revenue (INR) per brand. Vasant Kunj, 27 brands, 22:00; Galleria Market, 18 brands, 21:00 (date not printed). File: `data/market_data.csv`.

**Method.** Revenue = customers × units per customer × price per unit. Logs turn that product into a sum, so the variance of log revenue across brands splits into three parts. Each part's share is cov(log part, log revenue) / var(log revenue), the slope from regressing the log part on log revenue. The three shares add to one.

In [1]:
import json
import numpy as np, pandas as pd
from scipy import stats

df = pd.read_csv("data/market_data.csv")
df["units_per_customer"] = df.units / df.customers
df["price_per_unit"] = df.revenue_inr / df.units
df["revenue_per_customer"] = df.revenue_inr / df.customers
vk = df[df.event == "Vasant Kunj"].reset_index(drop=True)
ga = df[df.event == "Galleria Market"].reset_index(drop=True)
print(len(vk), "brands at Vasant Kunj;", len(ga), "at Galleria Market")

27 brands at Vasant Kunj; 18 at Galleria Market


**Data check.** The total printed on the Vasant Kunj leaderboard (534 customers, 1,067 units, INR 516,029) equals ranks 1 to 26 and leaves out Bumba, ranked 27th.

In [2]:
print("ranks 1-26:", vk.loc[vk["rank"] <= 26, ["customers", "units", "revenue_inr"]].sum().tolist())
print("all 27:   ", vk[["customers", "units", "revenue_inr"]].sum().tolist())

ranks 1-26: [534, 1067, 516029]
all 27:    [553, 1089, 519979]


**Figure 1 numbers.** Revenue per customer, Montrega against the other 26 brands.

In [3]:
rivals = vk[vk.brand != "Montrega"]
m_vk = vk[vk.brand == "Montrega"].iloc[0]
median_rpc = rivals.revenue_per_customer.median()
print(f"Montrega INR {m_vk.revenue_per_customer:,.0f} per customer; rival median INR {median_rpc:,.0f}; ratio {m_vk.revenue_per_customer / median_rpc:.2f}")
print("rivals with more customers:", (rivals.customers > m_vk.customers).sum(), "| rival median customers:", rivals.customers.median())

Montrega INR 6,653 per customer; rival median INR 836; ratio 7.96
rivals with more customers: 12 | rival median customers: 16.0


**Table 1.** Share of the cross-brand variance of log revenue carried by each part, plus the Spearman rank correlation between revenue and customers.

In [4]:
PARTS = ["customers", "units_per_customer", "price_per_unit"]

def shares(d):
    """Share of the variance of log revenue carried by each part; the three add to 1."""
    y = np.log(d.revenue_inr)
    return pd.Series({p: np.cov(np.log(d[p]), y)[0, 1] / y.var() for p in PARTS})

def spearman(d):
    rho, p = stats.spearmanr(d.revenue_inr, d.customers)
    return pd.Series({"spearman_rho": rho, "spearman_p": p})

samples = {"Vasant Kunj, 27 brands": vk, "Vasant Kunj, without Montrega": rivals, "Galleria Market, 18 brands": ga}
table1 = pd.DataFrame({k: pd.concat([shares(d), spearman(d)]) for k, d in samples.items()}).T
table1.round(3)

,customers,units_per_customer,price_per_unit,spearman_rho,spearman_p
"Vasant Kunj, 27 brands",0.251,0.240,0.509,0.373,0.056
"Vasant Kunj, without Montrega",0.364,0.056,0.580,0.399,0.044
"Galleria Market, 18 brands",0.033,-0.111,1.077,0.051,0.842


**Robustness.** Drop one brand at a time; then resample the 27 brands with replacement 10,000 times (bootstrap).

In [5]:
loo = pd.DataFrame([shares(vk.drop(index=i)) for i in vk.index])
print("leave one brand out, min and max:\n", loo.agg(["min", "max"]).round(3))

rng = np.random.default_rng(2026)
boot = pd.DataFrame([shares(vk.iloc[rng.integers(0, len(vk), len(vk))]) for _ in range(10_000)])
print("bootstrap 90% interval:\n", boot.quantile([0.05, 0.95]).round(3))
print("draws where price per unit > customers:", (boot.price_per_unit > boot.customers).mean().round(3))

leave one brand out, min and max:
      customers  units_per_customer  price_per_unit
min      0.185               0.056           0.427
max      0.364               0.302           0.580


bootstrap 90% interval:
       customers  units_per_customer  price_per_unit
0.05      0.034              -0.050           0.198
0.95      0.578               0.471           0.827
draws where price per unit > customers: 0.79


**Table 2.** Montrega at the two markets it won, and the 15 rival brands that sold at both.

In [6]:
m_ga = ga[ga.brand == "Montrega"].iloc[0]
cols = ["customers", "units_per_customer", "price_per_unit", "revenue_per_customer"]
table2 = pd.DataFrame({"Galleria": m_ga[cols], "Vasant Kunj": m_vk[cols]}).astype(float)
table2["ratio"] = table2["Vasant Kunj"] / table2["Galleria"]
print(table2.round(3))

both = vk.merge(ga, on="brand_id", suffixes=("_vk", "_ga")).query("brand_id != 'montrega'")
basket = both.units_per_customer_vk / both.units_per_customer_ga
print(len(both), "rivals at both | median basket ratio", round(basket.median(), 2), "| rose:", (basket > 1).sum(),
      "| median customer ratio", round((both.customers_vk / both.customers_ga).median(), 2))
print("largest rival rise:", both.loc[basket.idxmax(), "brand_vk"], round(basket.max(), 2))

                      Galleria  Vasant Kunj  ratio
customers               31.000       16.000  0.516
units_per_customer       1.516        6.125  4.040
price_per_unit        1677.660     1086.224  0.647
revenue_per_customer  2543.548     6653.125  2.616
15 rivals at both | median basket ratio 0.93 | rose: 5 | median customer ratio 0.46
largest rival rise: Indiya 2.14


**Checks a skeptic will ask for.** One customer bought more than ten bottles (stand team's recollection, no receipt). How much of the basket survives without that order? What would Montrega have earned with an ordinary basket? And how far below the price list did the realized price fall?

In [7]:
for x in [10, 12, 15, 20]:
    rest = (m_vk.units - x) / (m_vk.customers - 1)
    print(f"buyer of {x}: other 15 averaged {rest:.2f} bottles, {rest / m_ga.units_per_customer:.2f}x the Galleria basket")

cf = m_vk.customers * rivals.units_per_customer.median() * m_vk.price_per_unit
print(f"rival median basket, own customers and price: INR {cf:,.0f}, rank {(rivals.revenue_inr > cf).sum() + 1} of 27")

floor = 2200 / 2  # cheapest listed price per bottle: two 30 ml bottles for INR 2,200
print("realized INR per bottle:", round(m_vk.price_per_unit), "| revenue at the floor:", m_vk.units * floor,
      "| gap:", m_vk.units * floor - m_vk.revenue_inr)
print("price per bottle on a 12-bottle order if all other bottles sold at the floor:",
      round((m_vk.revenue_inr - (m_vk.units - 12) * floor) / 12, 1))

buyer of 10: other 15 averaged 5.87 bottles, 3.87x the Galleria basket
buyer of 12: other 15 averaged 5.73 bottles, 3.78x the Galleria basket
buyer of 15: other 15 averaged 5.53 bottles, 3.65x the Galleria basket
buyer of 20: other 15 averaged 5.20 bottles, 3.43x the Galleria basket
rival median basket, own customers and price: INR 22,530, rank 6 of 27
realized INR per bottle: 1086 | revenue at the floor: 107800.0 | gap: 1350.0
price per bottle on a 12-bottle order if all other bottles sold at the floor: 987.5


In [8]:
results = {"table1": table1.round(4).to_dict(orient="index"), "loo": loo.agg(["min", "max"]).round(4).to_dict(),
           "boot90": boot.quantile([0.05, 0.95]).round(4).to_dict(), "table2": table2.round(4).to_dict(orient="index")}
json.dump(results, open("output/results.json", "w"), indent=1)